In [ ]:
# =============================================================================
# Sleep English · Colab 一键运行（单单元格 · 默认 CPU · 配置/音色/成品全落 Drive）
# =============================================================================
# 一个单元格跑完：拉最新代码 → 装依赖 → 挂 Drive → 配置落 Drive → Kokoro 全部音色
# 与试听落 Drive → 起 Web 控制台（Cloudflare 免费域名 + 令牌闸门）→ 出片 → 归档。
# notebook 只是薄引导层：真正的逻辑在仓库 colab/ 的脚本里，所以仓库一更新，重跑本格
# 就吃到新版本。
#
# 首次使用
#   1. Runtime 保持默认 **CPU 即可**（本格默认强制 CPU 跑 Kokoro：结果与机器无关、
#      免费档也不用抢 GPU。想用 GPU：把下面 FORCE_CPU 改成 False，并把运行时切成 T4）
#   2. 左侧 🔑 Secrets 按需建键（键名在下面参数区指定）：
#      GEMINI_API_KEY 或 OPENAI_API_KEY —— LLM 生成脚本用（只用预生成脚本出片可不配）
#      CF_TUNNEL_TOKEN                —— 想用固定域名才需要
#      仓库是公开的，**不需要** GITHUB_TOKEN（私有 fork 才填 SECRET_GITHUB_TOKEN）
#   3. 从上往下就是全部步骤，直接运行本格；中途失败重跑本格即可（每步都幂等）
#   4. Colab 空闲约 90 分钟回收 VM：/content 下的东西（含 pip 装好的依赖）全丢，
#      重跑本格即可恢复；Drive 上的配置/音色/预生成脚本/成品都不受影响。
#      只重跑「配置/启动」而不跑安装步，出片时就会报 No module named 'kokoro'。
#
# 参数速查（只改下面参数区）
#   STEPS          —— 跑哪些步骤："all"（默认）或 ["voices"] / ["console"] / ["full"]…
#                     可单独重跑某一步：["config","voices"] 之类
#   RUN_SMOKE/RUN_FULL —— "all" 时是否顺带命令行出片（冒烟 10 组 / 参数区正式一期）
#   SLEEP_PAIRS    —— 正式一期组数：200 组 = 400 句 ≈ 100 分钟（CPU 约 1.5-3 小时）
#   FORCE_CPU      —— True = 强制 CPU（默认）；False = 有 GPU 就用 GPU
#   FETCH_ALL_VOICES / MAKE_PREVIEWS —— 28 个音色权重 + 试听 mp3 落 Drive
#   ARCHIVE        —— 出片后自动归档到 Drive（控制台里跑完也会自动归档）
#   KEEP_ALIVE     —— 长跑防掉线（best-effort：需要浏览器标签页保持打开）
#   STOP_ALL       —— True = 收尾时停掉 Web 服务与隧道
#
# 安全提醒：带 ?ct=令牌 的链接等于完整控制权（能读配置里的密钥、能用你的额度跑视频）。
#           不要转发、不要贴到公开 issue。
# =============================================================================

# ------------------------------- 参数区（一般只改这里） -----------------------
REPO_URL = "https://github.com/collinsgraciano/sleep_english.git"   # 公开仓库
BRANCH = "master"
REPO_DIR = "/content/sleep_english"

# 私有 fork 才需要：Secrets 里存 GitHub PAT 的**键名**；公开仓库留空 ""
SECRET_GITHUB_TOKEN = ""

# Drive：配置 / 音色 / 预生成脚本 / 成品都落这个目录（MyDrive 下的子目录名）
DRIVE_DIR = "sleep_english_colab"
# 渲染固定在容器本地盘：一期上千个卡片/音频小文件，写 Drive FUSE 慢到不可用
LOCAL_OUTPUT_DIR = "/content/sleep_english_output"

# LLM 通道（Step 0 生成脚本用）：填 Secrets 里的**键名**，不是密钥本身
SECRET_GEMINI_API_KEY = "GEMINI_API_KEY"
SECRET_OPENAI_API_KEY = ""
OPENAI_BASE_URL = ""                     # 例：https://your-endpoint/v1
OPENAI_MODEL = ""                        # 例：gemini-3.6-flash
GEMINI_MODEL = ""                        # 留空 = 用配置里的默认模型
COLAB_HF_ENDPOINT = ""                   # 留空 = huggingface.co；抓不动可填 https://hf-mirror.com

# 公网访问（Cloudflare 免费域名）
SECRET_CF_TUNNEL_TOKEN = ""              # 留空 = quick tunnel（每次开机换新域名，免费、无需 CF 账号）
CF_DOMAIN = ""                           # named tunnel 时才填（只为打印链接），例：sleep.example.com

# 本期视频参数（只在**首次生成配置**时生效；之后以 Drive 上的配置为准）
TOPIC = ""                               # 留空 = 从主题库随机挑
SLEEP_PAIRS = 200                        # A/B 组数：200 组 ≈ 100 分钟；CPU 上先跑 50 组更稳
NO_4K = True                             # CPU 上 4K 放大很慢，默认关
CHANNEL_NAME = ""                        # 留空 = 用配置里的频道名

# 执行控制
STEPS = "all"                            # "all" 或步骤名列表，见文件头
FORCE_CPU = True                         # True = 强制 CPU（默认）
FETCH_ALL_VOICES = True                  # 补齐 28 个英语音色（+中文兜底）权重
MAKE_PREVIEWS = True                     # 生成全部音色试听 mp3（首次约 3-6 分钟）
RUN_CONSOLE = True                       # 起 Web 控制台并打印公网链接
RUN_SMOKE = False                        # "all" 时顺带冒烟 10 组
RUN_FULL = False                         # "all" 时顺带正式跑参数区组数
ARCHIVE = True                           # 出片后（含控制台跑完）自动归档
TAIL_LOG_LINES = 80                      # 收尾打印多少行 uvicorn 日志
FORCE_INSTALL = False                    # True = 忽略安装标记，强制重装依赖
KEEP_ALIVE = False                       # True = 注入防掉线脚本（best-effort）
STOP_ALL = False                         # True = 收尾停掉 Web 服务与隧道
# -----------------------------------------------------------------------------

import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from urllib.parse import quote, urlparse, urlunparse

ALL_STEPS = ["code", "setup", "drive", "config", "voices", "console",
             "smoke", "full", "archive", "ops"]
SELECTED = set(ALL_STEPS) if str(STEPS).strip().lower() == "all" else set(
    s.strip() for s in ([STEPS] if isinstance(STEPS, str) else STEPS) if s.strip())
UNKNOWN = SELECTED - set(ALL_STEPS)
if UNKNOWN:
    raise SystemExit(f"STEPS 里有未知步骤：{sorted(UNKNOWN)}；可选：{ALL_STEPS}")

ENV_FILE = "/content/colab_env.sh"
PORT = "8766"
LOG_DIR = "/content/colab_logs"
DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_DIR}"
OUTPUT_DIR = LOCAL_OUTPUT_DIR
_SETUP_FLAG = "/content/.colab_setup_done"

_STEP_STATE = []          # [(name, ok, seconds, note)]
LLM_ENV = {}              # config 步写入；出片步复用


def _say(msg=""):
    print(msg, flush=True)


def _banner(title):
    _say("")
    _say("=" * 72)
    _say(f"  {title}")
    _say("=" * 72)


def _secret(key_name):
    """从 Colab Secrets 取密钥；非 Colab 环境回落同名环境变量。"""
    if not key_name:
        return ""
    try:
        from google.colab import userdata
        return (userdata.get(key_name) or "").strip()
    except Exception:
        return (os.environ.get(key_name, "") or "").strip()


_GH_TOKEN = _secret(SECRET_GITHUB_TOKEN)
_MASKS = [m for m in (_GH_TOKEN, quote(_GH_TOKEN, safe="")) if m]


def _mask(text):
    """git 失败时会把整条 URL（含令牌）回显出来，打印前统一打码。"""
    for m in _MASKS:
        text = str(text).replace(m, "****")
    return text


def _run(cmd, cwd=None, env=None, timeout=None, label=None, stream=False):
    """跑一条命令；默认捕获输出并打码后打印，stream=True 时直接实时打印到单元格。"""
    _say((f"$ [{label}] " if label else "$ ") + _mask(" ".join(cmd)))
    if stream:
        # 实时输出（装依赖 / 下音色可能几分钟，必须能看见进度）
        return subprocess.run(cmd, cwd=cwd, env=env, timeout=timeout)
    r = subprocess.run(cmd, cwd=cwd, env=env, text=True, encoding="utf-8",
                       errors="replace", capture_output=True, timeout=timeout)
    for out in (r.stdout, r.stderr):
        if out and out.strip():
            _say(_mask(out.rstrip()))
    return r


def _step(name, func, critical=False):
    """跑一个步骤并记时；非关键步骤失败只记录，最后统一汇报。"""
    if name not in SELECTED:
        return False
    _banner(f"[{name}]")
    t0 = time.time()
    try:
        note = func() or ""
        ok = True
    except SystemExit:
        raise
    except Exception as e:
        ok, note = False, f"{type(e).__name__}: {e}"
        _say(f"!! [{name}] 失败：{note}")
        if critical:
            _STEP_STATE.append((name, False, time.time() - t0, note))
            raise
    _STEP_STATE.append((name, ok, time.time() - t0, note))
    _say(f"--> [{name}] {'OK' if ok else 'FAILED'}（{time.time() - t0:.1f}s）{note}")
    return ok


# ---------------------------------------------------------------- 环境变量
env_vars = {
    "REPO_DIR": REPO_DIR,
    "PORT": PORT,
    "ENV_FILE": ENV_FILE,
    "DRIVE_ROOT": DRIVE_ROOT,
    "COLAB_OUTPUT_DIR": OUTPUT_DIR,
    "COLAB_TTS_ENGINE": "kokoro",
    "COLAB_TOPIC": TOPIC,
    "COLAB_SLEEP_PAIRS": str(SLEEP_PAIRS),
    "COLAB_NO_4K": "1" if NO_4K else "0",
    "COLAB_CHANNEL_NAME": CHANNEL_NAME,
    "CF_TUNNEL_TOKEN": _secret(SECRET_CF_TUNNEL_TOKEN),
    "CF_DOMAIN": CF_DOMAIN,
    "KOKORO_DEVICE": "cpu" if FORCE_CPU else "",
    "COLAB_SETUP_FLAG": _SETUP_FLAG,
    "COLAB_AUTO_ARCHIVE": "1",
    "COLAB_SCRIPTS_SYNC": "1",
    "COLAB_HF_ENDPOINT": COLAB_HF_ENDPOINT,
    "COLAB_LOG_DIR": LOG_DIR,
}
os.environ.update({k: str(v) for k, v in env_vars.items() if str(v)})
if not FORCE_CPU:
    os.environ.pop("KOKORO_DEVICE", None)       # 空值 = 有 GPU 就用 GPU


def _write_env_file():
    """把变量落成 shell 片段：serve.sh / 命令行 python 都 source 它。"""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(ENV_FILE, "w", encoding="utf-8") as fh:
        for k, v in env_vars.items():
            fh.write(f"export {k}={shlex.quote(str(v))}\n")


# ---------------------------------------------------------------- 各步骤实现
def _ensure_repo():
    """单独重跑某一步时兜底：仓库还没克隆就先补克隆（否则 setup/voices 找不到脚本）。"""
    if os.path.isdir(os.path.join(REPO_DIR, ".git")) or "code" in SELECTED:
        return
    _banner("[code] 前置：仓库还没克隆，自动补跑")
    step_code()


def _deps_ok():
    """本会话能不能 import kokoro（Colab 空闲回收/重连后 pip 包会丢，必须每次实检）。"""
    return subprocess.run([sys.executable, "-c", "import kokoro, misaki"],
                          capture_output=True).returncode == 0


def _ensure_deps(what="本步"):
    """TTS 相关步骤的前置：没装 kokoro 就先跑安装步（别等出片到 Step 2 才炸）。"""
    if _deps_ok():
        return
    _say(f"!! {what} 需要 Kokoro，但本会话没装 —— Colab 重连后 pip 包会丢，先补装依赖")
    step_setup()
    if not _deps_ok():
        raise RuntimeError("依赖仍不可用：看上面 pip 输出；要强制重装把 FORCE_INSTALL 设 True")


def step_setup():
    _ensure_repo()
    if FORCE_INSTALL:
        os.environ["COLAB_FORCE_INSTALL"] = "1"
    script = os.path.join(REPO_DIR, "colab", "setup.sh")
    r = _run(["bash", script], env={**os.environ}, timeout=5400,
             label="setup", stream=True)
    if r.returncode != 0:
        raise RuntimeError(f"setup.sh 退出码 {r.returncode}（看上面的自检输出）")
    return "依赖就绪"


def _auth_url(url):
    """私有 fork：把 PAT 放进 URL 的 userinfo 段；只活在内存里，不写进 .git/config。"""
    if not _GH_TOKEN:
        return url
    p = urlparse(url)
    return urlunparse(p._replace(netloc=f"{quote(_GH_TOKEN, safe='')}@{p.netloc}"))


def step_code():
    """拉最新代码：能 ff-only pull 就 pull，工作树被改过（configs 已 symlink 到 Drive）
    就整目录浅克隆 —— 持久物都在 Drive，重克隆无损。"""
    need_clone = not os.path.isdir(os.path.join(REPO_DIR, ".git"))
    if not need_clone:
        r = _run(["git", "-C", REPO_DIR, "pull", "--ff-only", _auth_url(REPO_URL), BRANCH])
        if r.returncode != 0:
            _say("!! pull 失败（工作树被改过，例如 configs 已 symlink 到 Drive）—— 整目录重克隆")
            _say("   持久物都在 Drive（配置 / 音色 / 预生成脚本 / 成品），重克隆不会丢东西")
            shutil.rmtree(REPO_DIR, ignore_errors=True)
            need_clone = True
    if need_clone:
        os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
        r = _run(["git", "clone", "--depth", "1", "-b", BRANCH,
                  _auth_url(REPO_URL), REPO_DIR])
        if r.returncode != 0:
            raise RuntimeError("克隆失败：公开仓库请检查参数区 REPO_URL / BRANCH；"
                               "私有 fork 需要 SECRET_GITHUB_TOKEN")
        if _GH_TOKEN:
            _run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
    head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                          text=True, encoding="utf-8", errors="replace",
                          capture_output=True).stdout.strip()
    return f"{'带令牌' if _GH_TOKEN else '匿名'} · {head}"


def step_drive():
    """挂 Drive + 写 env 片段。Drive 挂不上不阻塞（配置只在本次会话有效）。"""
    _write_env_file()
    mounted = os.path.isdir("/content/drive/MyDrive")
    if not mounted:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            mounted = os.path.isdir("/content/drive/MyDrive")
        except Exception as e:
            _say(f"!! Drive 挂载失败（{type(e).__name__}: {e}）")
    if mounted:
        os.makedirs(DRIVE_ROOT, exist_ok=True)
        return f"已挂载 {DRIVE_ROOT}"
    _say("!! 没挂 Drive：配置/音色/成品的持久化会失效（本会话仍能出片）")
    return "未挂载（降级）"


def step_config():
    """配置落 Drive + 平台归一化（密钥从 Secrets 读，不落仓库、不打印明文）。"""
    global LLM_ENV
    _ensure_repo()
    gemini_key = _secret(SECRET_GEMINI_API_KEY)
    openai_key = _secret(SECRET_OPENAI_API_KEY)
    if openai_key:
        if not (OPENAI_BASE_URL and OPENAI_MODEL):
            raise SystemExit("用自定义端点就要在参数区填 OPENAI_BASE_URL 和 OPENAI_MODEL")
        LLM_ENV = {"LLM_PROVIDER_TYPE": "openai", "OPENAI_API_KEY": openai_key,
                   "OPENAI_BASE_URL": OPENAI_BASE_URL, "OPENAI_MODEL": OPENAI_MODEL}
    elif gemini_key:
        LLM_ENV = {"LLM_PROVIDER_TYPE": "gemini", "GEMINI_API_KEY": gemini_key,
                   "GEMINI_MODEL": GEMINI_MODEL}
    else:
        _say("⚠️ 没读到 LLM 密钥：沿用 Drive 配置里已有的通道"
             "（若也没有，Step 0 生成脚本会失败；用预生成脚本出片则不需要密钥）")

    r = _run([sys.executable, os.path.join(REPO_DIR, "colab", "drive_config.py")],
             env={**os.environ}, label="drive_config")
    if r.returncode != 0:
        raise RuntimeError("配置持久化失败（看上面 drive_config 的输出）")

    r = _run([sys.executable, os.path.join(REPO_DIR, "colab", "colab_config.py")],
             env={**os.environ, **LLM_ENV}, label="colab_config")
    if r.returncode != 0:
        raise RuntimeError("配置写入失败（看上面 colab_config 的输出）")
    return "配置已落 Drive"


def step_voices():
    """Kokoro 全部音色 + 试听落 Drive（幂等：已有的直接跳过）。"""
    _ensure_repo()
    _ensure_deps("音色同步")
    cmd = [sys.executable, os.path.join(REPO_DIR, "colab", "kokoro_voices.py"), "--model"]
    if FETCH_ALL_VOICES:
        cmd += ["--sync-in", "--all"]
    if MAKE_PREVIEWS:
        cmd += ["--previews"]
    if FETCH_ALL_VOICES:
        cmd += ["--sync-back"]
    r = _run(cmd, env={**os.environ}, timeout=7200, label="kokoro_voices", stream=True)
    if r.returncode != 0:
        _say("!! 音色步骤没跑完：不影响出片（缺哪个音色会在用到时现场下载）")
        return "未完全就绪（不阻塞出片，重跑本格会接着补）"
    return "音色/试听已同步"


def _detect_public_url(text):
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text or "")
    if m:
        return m.group(0)
    m = re.search(r"https://[^\s]+", text or "")
    return m.group(0) if m else ""


def step_console():
    """起 Web 控制台（令牌闸门 + Cloudflare 隧道）并打印带令牌链接。"""
    if not RUN_CONSOLE:
        return "跳过（RUN_CONSOLE=False）"
    _ensure_repo()
    _ensure_deps("Web 控制台")
    _say("启动 Web 服务 + Cloudflare 隧道（约 10-40 秒）…")
    r = _run(["bash", os.path.join(REPO_DIR, "colab", "serve.sh")],
             env={**os.environ}, timeout=600, label="serve")
    url = _detect_public_url(r.stdout)
    with open(ENV_FILE, "r", encoding="utf-8") as fh:
        token = ""
        for line in fh:
            if line.startswith("export COLAB_ACCESS_TOKEN="):
                token = line.split("=", 1)[1].strip().strip("'\"")
    if url:
        _say("")
        _say("*" * 72)
        _say("  Web 控制台（和本机 localhost:8766 一样，改配置直接落 Drive）：")
        _say(f"    {url}/?ct={token}")
        _say("  链接等于完整控制权，别转发；域名每次开机都会换，令牌不变。")
        _say("*" * 72)
        return f"已启动 {url}"
    _say("!! 没拿到公网 URL（看 /content/colab_logs/cloudflared.log）")
    _say("   本地服务仍在跑：http://127.0.0.1:8766（也可直接用命令行出片）")
    return "已启动（无公网 URL）"


def _run_pipeline(topic, pairs, extra_note=""):
    _ensure_repo()
    _ensure_deps("出片")
    cmd = [sys.executable, "pipeline.py", "--cefr", "A2",
           "--sleep-pairs", str(pairs), "--output", OUTPUT_DIR]
    if topic:
        cmd += ["--topic", topic]
    if NO_4K:
        cmd += ["--no-4k"]
    if CHANNEL_NAME:
        cmd += ["--sleep-channel-name", CHANNEL_NAME]
    _say(f"预计：{'冒烟 10 组约 5-10 分钟' if pairs <= 10 else f'{pairs} 组在 CPU 上约 1.5-3 小时'}"
         f"（{extra_note}）")
    r = subprocess.run(cmd, cwd=os.path.join(REPO_DIR, "pipeline"),
                       env={**os.environ, **LLM_ENV}, text=False)
    if r.returncode != 0:
        raise RuntimeError(f"pipeline.py 退出码 {r.returncode}（中断后重跑本格会文件级续传）")
    return f"{pairs} 组完成"


def step_smoke():
    if not RUN_SMOKE:
        return "跳过（RUN_SMOKE=False）"
    return _run_pipeline("Ordering Coffee", 10, "主题固定，验证全链路")


def step_full():
    if not RUN_FULL:
        return "跳过（RUN_FULL=False）"
    return _run_pipeline(TOPIC, SLEEP_PAIRS, "用参数区 TOPIC / SLEEP_PAIRS")


def step_archive():
    if not ARCHIVE:
        return "跳过（ARCHIVE=False）"
    _ensure_repo()
    r = _run([sys.executable, os.path.join(REPO_DIR, "colab", "archive_to_drive.py")],
             env={**os.environ}, timeout=3600, label="archive")
    if r.returncode != 0:
        _say("!! 归档没成功：Drive 挂了吗？成品只在本会话的容器本地盘")
    return "成品 + 状态已备份"


def step_ops():
    """收尾：日志尾巴 + 成品清单（本地/Drive）+ 可选停止服务。"""
    log_path = os.path.join(LOG_DIR, "uvicorn.log")
    if os.path.isfile(log_path):
        with open(log_path, "r", encoding="utf-8", errors="replace") as fh:
            lines = fh.readlines()[-int(TAIL_LOG_LINES or 0):] if TAIL_LOG_LINES else []
        if lines:
            _say(f"--- {log_path} 最后 {len(lines)} 行")
            _say("".join(lines).rstrip())
    for label, root in (("本地", OUTPUT_DIR), ("Drive", os.path.join(DRIVE_ROOT, "output"))):
        if not root or not os.path.isdir(root):
            continue
        hits = []
        for dirpath, dirs, files in os.walk(root):
            dirs[:] = [d for d in dirs
                       if d not in ("audio", "cards", "clips", "images", "tmp_sleep_blocks")]
            for f in files:
                if f.endswith(".mp4"):
                    p = os.path.join(dirpath, f)
                    hits.append((os.path.getsize(p), p))
        _say(f"--- {label}：{root}（{len(hits)} 个 mp4）")
        for size, p in sorted(hits, reverse=True)[:20]:
            _say(f"  {size / 1024 / 1024:9.1f} MB  {os.path.relpath(p, root)}")
    if STOP_ALL:
        subprocess.run("pkill -f 'uvicorn secure_gate:app'; pkill -f 'cloudflared tunnel'",
                       shell=True)
        _say("已停止 Web 服务与隧道")
    else:
        _say("Web 服务与隧道仍在运行（本格跑完不影响）；要停掉把 STOP_ALL 设 True 重跑本格")
    return "收尾完成"


def _install_keep_alive():
    """长跑防掉线（best-effort）：Colab 空闲约 90 分钟回收 VM，长任务要压住这个计时器。

    注意这是社区常用的 DOM 点击法：只在浏览器标签页保持打开时有效，Colab 前端改版
    可能失效 —— 所以默认关闭，长任务想开再开。
    """
    if not KEEP_ALIVE:
        return
    try:
        from IPython.display import Javascript, display
        display(Javascript("""
function __dshKeepAlive(){
  const b = document.querySelector("colab-connect-button");
  if (b) b.click();
  console.log("keep-alive", new Date().toISOString());
}
setInterval(__dshKeepAlive, 60000);
"""))
        _say("已注入防掉线脚本（best-effort：标签页要一直开着；Colab 前端改版后可能失效）")
    except Exception as e:
        _say(f"防掉线脚本注入失败（不影响运行）：{e}")


# ---------------------------------------------------------------- 主流程
_banner(f"Sleep English · Colab 一键运行 · 步骤：{sorted(SELECTED)}")
_say(f"CPU 强制：{'是（KOKORO_DEVICE=cpu）' if FORCE_CPU else '否（有 GPU 就用 GPU）'}"
     f" · Drive：{DRIVE_ROOT} · 输出：{OUTPUT_DIR}")
_install_keep_alive()

failed = []
try:
    _step("code", step_code, critical=True)
    _step("setup", step_setup, critical=True)
    _step("drive", step_drive, critical=True)
    _step("config", step_config, critical=True)
    _step("voices", step_voices)
    _step("console", step_console)
    _step("smoke", step_smoke)
    _step("full", step_full)
    _step("archive", step_archive)
    _step("ops", step_ops)
except SystemExit:
    raise
except BaseException as e:
    _say(f"!! 关键步骤失败，后续步骤不再执行：{type(e).__name__}: {e}")

_banner("执行汇总")
for name, ok, secs, note in _STEP_STATE:
    _say(f"  {'✔' if ok else '✖'} {name:<8} {secs:7.1f}s  {note}")
    if not ok:
        failed.append(name)
_skipped = [s for s in ALL_STEPS if s not in SELECTED]
if _skipped:
    _say(f"  ⏭ 未选步骤：{', '.join(_skipped)}（改参数区 STEPS 可单独跑）")
if failed:
    _say(f"!! 失败步骤：{', '.join(failed)} —— 修好后重跑本格即可（每步幂等）")
else:
    _say("全部所选步骤完成。控制台里改的任何配置都直接落 Drive，下回开机自动恢复。")